<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 01: 環境セットアップと検証</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Microsoft Foundry Agent Observability
  </p>
</div>

## 学習内容

このラボでは、**Microsoft Foundry** と **Azure AI Projects SDK** を使用できるよう開発環境を設定します。これは、フライト・ホテル・レンタカーの手配を支援するマルチエージェント旅行アシスタント **Contoso Travel** エージェントシステムを構築するための基盤です。

---


## 1. 依存関係のインストール

Azure AI Projects SDK、認証ライブラリ、トレーシング用 OpenTelemetry、旅行データ操作用 pandas が必要です。

---


In [ ]:
# azure-ai-projects: エージェント・評価・テレメトリ用 Foundry SDK
# azure-identity: DefaultAzureCredential 認証チェーン
# opentelemetry + azure-monitor: 分散トレーシング（Lab 04 以降）
%pip install azure-ai-projects>=2.0.0 azure-identity python-dotenv opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry pandas --quiet

## 2. 環境変数の設定

`labs/notebooks/` にある `.env` ファイルを作成し、Foundry ポータルの値と一致しているか確認してください:

| 変数名 | 取得場所 |
|--------|---------|
| `AZURE_AI_PROJECT_ENDPOINT` | Foundry ポータル → プロジェクト概要 |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | Foundry ポータル → モデル + エンドポイント → 名前列 |
| `MODEL_ENDPOINT` | モデル + エンドポイント → デプロイをクリック → ターゲット URI（ベース URL のみ） |
| `MODEL_API_KEY` | モデル + エンドポイント → デプロイをクリック → キーを表示 |

---


In [ ]:
# 共有 .env から環境変数を読み込んで検証
import os
from dotenv import load_dotenv

# abspath(".") = ノートブックの CWD（例: labs/notebooks/1-prompt-agents/）
# dirname() で一つ上のディレクトリ → .env が存在する labs/notebooks/ へ。
# このパターンにより、すべてのラボサブディレクトリが単一の .env を共有できます。
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

required_vars = [
    "AZURE_AI_PROJECT_ENDPOINT",
    "AZURE_AI_MODEL_DEPLOYMENT_NAME",
]

# オプション変数は Lab 06（レッドチーミング）でのみ必要
optional_vars = [
    "MODEL_ENDPOINT",
    "MODEL_API_KEY",
]

print("✅ 必須環境変数を確認中...\n")
all_set = True
for var in required_vars:
    value = os.environ.get(var)
    if value:
        # 出力にシークレット全体が表示されないよう切り詰め
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ❌ {var} が設定されていません")
        all_set = False

print("\n📋 オプション環境変数を確認中...\n")
for var in optional_vars:
    value = os.environ.get(var)
    if value:
        display = value[:12] + "..." + value[-4:] if len(value) > 20 else value
        print(f"  ✅ {var} = {display}")
    else:
        print(f"  ⚠️  {var} が設定されていません（レッドチーミング ラボで必要）")

if all_set:
    print("\n🎉 必須変数はすべて設定されています！")
else:
    print("\n⚠️  続行する前に、.env ファイルで不足している変数を設定してください。")

## 3. Azure 接続の検証

SDK を使用して Microsoft Foundry プロジェクトに接続できるか確認します。

---


In [ ]:
# SDK が Foundry プロジェクトエンドポイントに到達できるか検証
from azure.identity import DefaultAzureCredential, AzureCliCredential, InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]

# az CLI・マネージド ID・環境変数を自動的にチェーン
credential = DefaultAzureCredential() if not tenant_id else AzureCliCredential(tenant_id=tenant_id)
#credential = InteractiveBrowserCredential(tenant_id=tenant_id)
# エージェント・評価・テレメトリなど Foundry 機能すべての中心クライアント
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Microsoft Foundry に接続しました！")
print(f"   エンドポイント: {endpoint}")

## 4. OpenAI クライアントの検証

Azure AI Projects SDK は OpenAI 互換クライアントを提供します。簡単なテストプロンプトで動作を確認します。

---


In [ ]:
# get_openai_client() はプロジェクトエンドポイントと認証情報を継承した
# 完全設定済みの OpenAI クライアントを返します — 手動設定は不要
openai_client = project_client.get_openai_client()

model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# Foundry プロジェクトエンドポイントは Responses API をサポート
# （chat.completions は 404 になる場合があるため responses.create を使用）
response = openai_client.responses.create(
    model=model_name,
    input="Contoso Travelからこんにちは！",
)

print(f"✅ モデル '{model_name}' が応答しています！")
print(f"   応答: {response.output_text}")


## 5. Contoso Travel サンプルデータの確認

旅行エージェントはフライト・ホテル・レンタカーの CSV データファイルを使用します。エージェントが扱うデータをプレビューしましょう。

---


In [ ]:
# Contoso Travel フライトインベントリの読み込みとプレビュー
import pandas as pd

# ノートブックディレクトリからの相対パス → ../../data/ で labs/data/ に到達
data_path = "../../data/contoso-travel"

flights = pd.read_csv(f"{data_path}/flights.csv")
print(f"フライト: {len(flights)} 件")
print(f"ルート: {flights['origin'].nunique()} 出発地 → {flights['destination'].nunique()} 目的地")
print(f"価格帯: ${flights['price_usd'].min():.0f} - ${flights['price_usd'].max():.0f}")
flights.head(3)

In [ ]:
# 目的地都市のホテルインベントリをプレビュー
hotels = pd.read_csv(f"{data_path}/hotels.csv")
print(f"ホテル: {len(hotels)} 件")
print(f"都市: {', '.join(hotels['city'].unique())}")
print(f"星評価: {hotels['star_rating'].min()}-{hotels['star_rating'].max()} 星")
print(f"価格帯: ${hotels['price_per_night_usd'].min():.0f} - ${hotels['price_per_night_usd'].max():.0f}/泊")
hotels.head(3)

In [ ]:
# 都市・車種別のレンタカーオプションをプレビュー
cars = pd.read_csv(f"{data_path}/car_rentals.csv")
print(f"レンタカー: {len(cars)} 件")
print(f"都市: {', '.join(cars['city'].unique())}")
print(f"タイプ: {', '.join(cars['car_type'].unique())}")
print(f"価格帯: ${cars['price_per_day_usd'].min():.0f} - ${cars['price_per_day_usd'].max():.0f}/日")
cars.head(3)

In [ ]:
# リークを防ぐため HTTP 接続とトークンキャッシュを解放
# 順序重要: 親の project_client より先にリーフクライアントを閉じる
openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました。セットアップ完了！")

---